In [ ]:
!pip install ultralytics 'sng4onnx>=1.0.1' 'onnx_graphsurgeon>=0.3.26' 'ai-edge-litert>=1.2.0' 'onnx>=1.12.0,<2.0.0' 'onnx2tf>=1.26.3,<1.29.0' 'onnxslim>=0.1.71' 'onnxruntime'

In [ ]:
!pip install -U ultralytics "ray[tune]"

In [ ]:
!pip install -U ultralytics wandb

input link ke dataset

In [4]:
!mkdir "/content/dataset"

mkdir: cannot create directory ‘/content/dataset’: File exists


In [ ]:
#!yolo settings tensorboard=True
!yolo settings wandb=True
#%load_ext tensorboard
#%tensorboard --logdir "/tmp/ray"

In [6]:
from ultralytics import YOLO
import wandb as wb
from ray import tune
import os
from google.colab import userdata
from datetime import datetime, timezone


Di tab kanan ada secrets, centang notebook access

In [7]:
class WandbHandler:
  def __init__(self, project_name, root='/content/'):
    self.project_name = project_name
    self.root = root
    wb.login(key=userdata.get('WANDB_API_KEY'))

  def download_from_wandb(self, dataset_name):
    dataset_root = f"{self.root}dataset/{dataset_name}"
    run = wb.init(project=self.project_name, job_type="training")
    artifact = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
      return run, dataset_root

    artifact.download(root=dataset_root)
    return run, dataset_root



  def upload_to_wandb(self, dataset_dir):
    artifact = wb.Artifact(name=dataset_dir, type="dataset")
    artifact.add_dir(self.root + dataset_dir)
    with wb.init(project=self.project_name, job_type='upload_dataset') as run:
      run.log_artifact(artifact)

  def update_dataset(self, dataset_dir):
    with wb.init(project=self.project_name, job_type='update_dataset') as run:
      artifact = wb.Artifact(name=dataset_dir, type="dataset")
      artifact.add_dir(self.root + dataset_dir)
      run.log_artifact(artifact)


  def update_file(self, dataset_dir, file_name):
    with wb.init(project=self.project_name, job_type='update_file') as run:
      saved_artifact = run.use_artifact(dataset_dir+":latest")
      draft_artifact = saved_artifact.new_draft()

      draft_artifact.remove(saved_artifact.get_entry(file_name))
      draft_artifact.add_file(local_path= self.root + dataset_dir + "/" + file_name, name=file_name)

      draft_artifact.save()





In [8]:
_processed_plots = {}

def _custom_table(x, y, classes, title="Precision Recall Curve", x_title="Recall", y_title="Precision"):
    """Create and log a custom metric visualization table.

    This function crafts a custom metric visualization that mimics the behavior of the default wandb precision-recall
    curve while allowing for enhanced customization. The visual metric is useful for monitoring model performance across
    different classes.

    Args:
        x (list): Values for the x-axis; expected to have length N.
        y (list): Corresponding values for the y-axis; also expected to have length N.
        classes (list): Labels identifying the class of each point; length N.
        title (str, optional): Title for the plot.
        x_title (str, optional): Label for the x-axis.
        y_title (str, optional): Label for the y-axis.

    Returns:
        (wandb.Object): A wandb object suitable for logging, showcasing the crafted metric visualization.
    """
    import polars as pl  # scope for faster 'import ultralytics'
    import polars.selectors as cs

    df = pl.DataFrame({"class": classes, "y": y, "x": x}).with_columns(cs.numeric().round(3))
    data = df.select(["class", "y", "x"]).rows()

    fields = {"x": "x", "y": "y", "class": "class"}
    string_fields = {"title": title, "x-axis-title": x_title, "y-axis-title": y_title}
    return wb.plot_table(
        "wandb/area-under-curve/v0",
        wb.Table(data=data, columns=["class", "y", "x"]),
        fields=fields,
        string_fields=string_fields,
    )


def _plot_curve(
    x,
    y,
    names=None,
    id="precision-recall",
    title="Precision Recall Curve",
    x_title="Recall",
    y_title="Precision",
    num_x=100,
    only_mean=False,
):
    """Log a metric curve visualization.

    This function generates a metric curve based on input data and logs the visualization to wandb. The curve can
    represent aggregated data (mean) or individual class data, depending on the 'only_mean' flag.

    Args:
        x (np.ndarray): Data points for the x-axis with length N.
        y (np.ndarray): Corresponding data points for the y-axis with shape (C, N), where C is the number of classes.
        names (list, optional): Names of the classes corresponding to the y-axis data; length C.
        id (str, optional): Unique identifier for the logged data in wandb.
        title (str, optional): Title for the visualization plot.
        x_title (str, optional): Label for the x-axis.
        y_title (str, optional): Label for the y-axis.
        num_x (int, optional): Number of interpolated data points for visualization.
        only_mean (bool, optional): Flag to indicate if only the mean curve should be plotted.

    Notes:
        The function leverages the '_custom_table' function to generate the actual visualization.
    """
    import numpy as np

    # Create new x
    if names is None:
        names = []
    x_new = np.linspace(x[0], x[-1], num_x).round(5)

    # Create arrays for logging
    x_log = x_new.tolist()
    y_log = np.interp(x_new, x, np.mean(y, axis=0)).round(3).tolist()

    if only_mean:
        table = wb.Table(data=list(zip(x_log, y_log)), columns=[x_title, y_title])
        wb.log({title: wb.plot.line(table, x_title, y_title, title=title)})
    else:
        classes = ["mean"] * len(x_log)
        for i, yi in enumerate(y):
            x_log.extend(x_new)  # add new x
            y_log.extend(np.interp(x_new, x, yi))  # interpolate y to new x
            classes.extend([names[i]] * len(x_new))  # add class names
        wb.log({id: _custom_table(x_log, y_log, classes, title, x_title, y_title)}, commit=False)

def _log_plots(plots, step):
    """Log plots to WandB at a specific step if they haven't been logged already.

    This function checks each plot in the input dictionary against previously processed plots and logs new or updated
    plots to WandB at the specified step.

    Args:
        plots (dict): Dictionary of plots to log, where keys are plot names and values are dictionaries containing plot
            metadata including timestamps.
        step (int): The step/epoch at which to log the plots in the WandB run.

    Notes:
        The function uses a shallow copy of the plots dictionary to prevent modification during iteration.
        Plots are identified by their stem name (filename without extension).
        Each plot is logged as a WandB Image object.
    """
    for name, params in plots.copy().items():  # shallow copy to prevent plots dict changing during iteration
        timestamp = params["timestamp"]
        if _processed_plots.get(name) != timestamp:
            wb.log({name.stem: wb.Image(str(name))}, step=step)
            _processed_plots[name] = timestamp

def get_dir(filepath):
  dir = filepath.split("/")
  dir.pop(-1)
  dir = "/".join(dir)
  return dir

def custom_on_train_end(trainer):
    """Save the best model as an artifact and log final plots at the end of training."""
    _log_plots(trainer.validator.plots, step=trainer.epoch + 1)
    _log_plots(trainer.plots, step=trainer.epoch + 1)
    art_model = wb.Artifact(type="model", name=f"{trainer.args.name}_model")
    art_mobile = wb.Artifact(type="mobile-model", name=f"{trainer.args.name}_mobile")
    if trainer.best.exists():
        art_model.add_file(trainer.best)
        wb.run.log_artifact(art_model, aliases=["best"])
        best_model = YOLO(trainer.best)
        tflite = best_model.export(format="tflite", data=f"{trainer.args.data}", device="cpu")
        art_mobile.add_file(tflite)
        tflite_dir = get_dir(tflite)
        dataset_dir = get_dir(trainer.args.data)
        art_mobile.add_file(f"{dataset_dir}/hasil_gizi_100gram.csv")
        art_mobile.add_file(f"{tflite_dir}/metadata.yaml")
        wb.run.log_artifact(art_mobile, aliases=["best"])

    # Check if we actually have plots to save
    if trainer.args.plots and hasattr(trainer.validator.metrics, "curves_results"):
        for curve_name, curve_values in zip(trainer.validator.metrics.curves, trainer.validator.metrics.curves_results):
            x, y, x_title, y_title = curve_values
            _plot_curve(
                x,
                y,
                names=list(trainer.validator.metrics.names.values()),
                id=f"curves/{curve_name}",
                title=curve_name,
                x_title=x_title,
                y_title=y_title,
            )
    wb.run.finish()  # required or run continues on dashboard

In [ ]:
project = "FAIt (Food AI-based tracking)"
dataset_name = "Food_Computer_Vision_Dataset" # nama dataset yang suda ada di Weights & Biases
wandb_handler = WandbHandler(project)
run, dataset_dir = wandb_handler.download_from_wandb(dataset_name)

In [10]:

model_type = "yolo26n" # tipe model bisa yolo26n yolo26s yolo26m yolo26l yolo26x (dimana n adalah model yg paling kecil dan x adalah yang palih besar)
epoch = 1 # jumlah epoch
model = YOLO(f"{model_type}.yaml").load(f"{model_type}.pt")
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d-%H-%M')

Transferred 708/708 items from pretrained weights


In [11]:
def setup_custom_callbacks(trainer):
    # Filter the trainer's callbacks, not the model's
    trainer.callbacks["on_train_end"] = [
        f for f in trainer.callbacks["on_train_end"]
        if "ultralytics.utils.callbacks.wb" not in f.__module__
    ]

    # Add your custom callback to the trainer's lifecycle
    if custom_on_train_end not in trainer.callbacks["on_train_end"]:
        trainer.callbacks["on_train_end"].append(custom_on_train_end)

# Register the setup function to trigger right before training starts
model.add_callback("on_pretrain_routine_start", setup_custom_callbacks)

In [ ]:
result = model.train(
    data=f"{dataset_dir}/data.yaml",
    project=project,
    name=f"{model_type}_{dataset_name}_{epoch}_{timestamp}",
    epochs=epoch,
    val=True,
    imgsz=640,
    save=True,
    verbose=False,

)

In [13]:
best_model = YOLO(f"{result.save_dir}/weights/best.pt")
# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list containing mAP50-95 for each category

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,384,391 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1083.7±825.7 MB/s, size: 60.2 KB)
val: Scanning /content/dataset/Food_Computer_Vision_Dataset/valid/labels.cache... 549 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 549/549 153.5Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 63, len(boxes) = 1291. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 5.5it/s 6.4s
                   all        549       1291    0.00797      0.226       0.02     0.0133
               alpukat          3         21          0          0          0          0
           

array([          0,   0.0026152,   0.0019658,    0.018014,           0,           0,    0.021364,           0,    0.011364,   0.0029139,           0,    0.021871,   0.0095541,           0,           0,    0.011178,    0.013268,           0,    0.013268,           0,           0,           0,           0,    0.046045,
         0.0039476,     0.11397,           0,           0,   0.0064665,    0.043276,    0.013268,    0.032871,   0.0026889,           0,   0.0007435,    0.013268,           0,           0,           0,   0.0018273,    0.013268,   0.0035318,    0.022483,   0.0055172,    0.095512,           0,    0.017866,    0.072932,
          0.013268])

In [14]:
# search_space = {
#     "lr0": tune.uniform(1e-5, 1e-1),
# }
# def setup_custom_callbacks(trainer):
#     """Clears default W&B finish logic inside the Ray worker."""
#     trainer.remove_callback("on_train_end", wandb.run.finish) # Remove only finish if possible
#     trainer.add_callback("on_train_end", custom_on_train_end)

# # Register the setup function to trigger inside the Ray worker
# model.add_callback("on_pretrain_routine_start", setup_custom_callbacks)

# result = model.tune(
  # data=f"{dataset_dir}/data.yaml",
  # project=project,
  # name=f"{model_type}_{dataset_name}_{epoch}_{timestamp}",
#     gpu_per_trial=1,
#     epochs=epoch,
#     grace_period=5,
#     iterations=2,
#     plots=False,
#     save=True,
#     use_ray=True,
#     verbose=False,

# )


In [15]:

# # Get the best result path from the grid
# best_result = result.get_best_result(metric="metrics/mAP_50-95", mode="max")
# best_model_path = os.path.join(best_result.path, "weights", "best.pt")

# # Load this specific model
# model = YOLO(best_model_path)

# # Run validation - ensure 'data' points to the absolute path of your YAML
# metrics = model.val(data="/content/dataset/New_Food_Dataset/data.yaml")